# CCI Session 2 — Lab 6: Clinical Data Extraction with GPT-4.1-mini & Email Report

**Objective:** Use OpenAI's GPT-4.1-mini to extract structured clinical data from a patient discharge note, then email the JSON result via Gmail SMTP.

**Notebook follows the 9-section KHCC template.**

### Before you start
1. Click the **key icon** in Colab's left sidebar to open **Secrets**.
2. Add a secret named `OPENAI_API_KEY` with your OpenAI key.
3. (For email) Add `GMAIL_ADDRESS` and `GMAIL_APP_PASSWORD`.
   - Generate an App Password at https://myaccount.google.com/apppasswords

---
## Section 1: Package Installations

In [ ]:
!pip install openai -q

---
## Section 2: Imports

In [ ]:
import json
import csv
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime
from openai import OpenAI
from google.colab import userdata

---
## Section 3: Configuration

Fill in the `RECIPIENT_EMAIL` with the address that should receive the report.

In [ ]:
# OpenAI
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
MODEL = "gpt-4.1-mini"

# Gmail SMTP
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = userdata.get("GMAIL_ADDRESS")
SENDER_APP_PASSWORD = userdata.get("GMAIL_APP_PASSWORD")

# TODO: set the recipient email address
RECIPIENT_EMAIL = ""  # <-- put the target email here

REPORT_DATE = datetime.now().strftime("%Y-%m-%d %H:%M")
OUTPUT_CSV = "extracted_patient_data.csv"

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Config loaded — model: {MODEL}, date: {REPORT_DATE}")

---
## Section 4: Prompts

We need two prompts:
- A **system prompt** telling the model its role.
- An **extraction prompt** with the JSON schema and a `{note}` placeholder.

In [ ]:
SYSTEM_PROMPT = """You are a clinical data extraction assistant at King Hussein Cancer Center.
Extract structured data from the patient note provided by the user.
Return ONLY valid JSON with no markdown fences and no extra text."""

# TODO: Complete the extraction prompt.
# It should list every field you want extracted and include {note} where the
# patient note will be inserted.  Use the schema below as a guide.
#
# Fields to extract:
#   patient_name, mrn, age, gender, primary_diagnosis, stage,
#   hemoglobin, wbc, platelets, creatinine,
#   medications (list), allergies (list), procedures (list),
#   discharge_disposition, follow_up

EXTRACTION_PROMPT = """
Extract the following fields from the patient note below.
Return a single JSON object with exactly these keys:

TODO — paste the JSON schema here (see instructions above)

If a field is not mentioned in the note, use null.

--- PATIENT NOTE ---
{note}
--- END NOTE ---
"""

---
## Section 5: Functions

Implement three functions:
1. `extract_patient_data(note_text)` — calls GPT-4.1-mini and returns a dict
2. `build_email_body(data)` — formats the dict into a readable string
3. `send_email(recipient, subject, body)` — sends via Gmail SMTP

In [ ]:
def extract_patient_data(note_text: str) -> dict:
    """Send a patient note to GPT-4.1-mini and return structured JSON."""
    # TODO: call client.chat.completions.create() with:
    #   - model = MODEL
    #   - messages = system prompt + user prompt (with note inserted)
    #   - temperature = 0.0  (for deterministic extraction)
    # Then parse the response content as JSON and return the dict.
    pass


def build_email_body(data: dict) -> str:
    """Format extracted data into a readable email body."""
    # TODO: build a multi-line string that includes:
    #   - Report header with date
    #   - Patient name, MRN, age, gender
    #   - Diagnosis and stage
    #   - Lab values (hemoglobin, wbc, platelets, creatinine)
    #   - Medications, allergies, procedures
    #   - Discharge disposition and follow-up
    #   - The full JSON dump at the end
    pass


def send_email(recipient: str, subject: str, body: str):
    """Send a plain-text email via Gmail SMTP with TLS."""
    msg = MIMEMultipart()
    msg["From"] = SENDER_EMAIL
    msg["To"] = recipient
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    # TODO: open an SMTP connection, call starttls(), login(), send_message()
    # Hint: use smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as a context manager
    pass

---
## Section 6: Main Code — Sample Patient Note + Extraction

Run the extraction on the sample note below.

In [ ]:
SAMPLE_NOTE = """
DISCHARGE SUMMARY
King Hussein Cancer Center — Department of Medical Oncology

Patient Name: Ahmad Khalil Al-Rashid
MRN: P-20458
Age: 62 years   Sex: Male
Date of Admission: 2026-03-15
Date of Discharge: 2026-03-22

PRIMARY DIAGNOSIS: Non-Small Cell Lung Cancer (Adenocarcinoma), Stage IIIB

HISTORY OF PRESENT ILLNESS:
Mr. Al-Rashid is a 62-year-old male with a history of NSCLC diagnosed in January 2026
after presenting with persistent cough and hemoptysis. CT chest showed a 4.2 cm right
upper lobe mass with mediastinal lymphadenopathy. Biopsy confirmed adenocarcinoma,
EGFR wild-type, ALK negative, PD-L1 TPS 65%.

HOSPITAL COURSE:
Patient admitted for Cycle 1 of Pembrolizumab 200 mg IV + Carboplatin AUC 5 +
Pemetrexed 500 mg/m2. Tolerated infusion well. Developed Grade 1 nausea managed
with Ondansetron. No febrile neutropenia.

LABORATORY DATA (Discharge):
  Hemoglobin: 11.2 g/dL
  WBC: 5.8 K/uL
  Platelets: 198 K/uL
  Creatinine: 0.9 mg/dL
  ALT: 22 U/L
  AST: 18 U/L

MEDICATIONS AT DISCHARGE:
1. Ondansetron 8 mg PO PRN nausea
2. Dexamethasone 4 mg PO daily x 3 days
3. Pantoprazole 40 mg PO daily
4. Filgrastim 300 mcg SC daily x 5 days (if ANC < 1.5)

ALLERGIES: Sulfonamides (rash), Ibuprofen (GI bleeding)

PROCEDURES DURING ADMISSION:
- Port-a-Cath insertion (right subclavian, IR-guided)
- CT-guided core biopsy of right upper lobe mass (prior admission)

DISCHARGE DISPOSITION: Home in stable condition

FOLLOW-UP:
Medical Oncology clinic in 21 days for Cycle 2. CBC and CMP 7 days prior.
CT chest/abdomen/pelvis after Cycle 3 for response assessment.

Attending Physician: Dr. Nadia Haddad, MD
Department of Medical Oncology, KHCC
"""

print("Sending note to GPT-4.1-mini for extraction...")
extracted = extract_patient_data(SAMPLE_NOTE)
print("\nExtracted JSON:")
print(json.dumps(extracted, indent=2, ensure_ascii=False))

---
## Section 7: Build CSV

Flatten list fields into comma-separated strings and save one row to CSV.

In [ ]:
# TODO: create a flat dict where list values are joined with ", "
# Then write it to OUTPUT_CSV using csv.DictWriter

# flat = ...
# with open(OUTPUT_CSV, "w", newline="") as f:
#     ...

# print(f"Saved to {OUTPUT_CSV}")

---
## Section 8: Email Results

Build the email subject and body, preview it, then send.

In [ ]:
subject = f"[KHCC-CCI] Patient Data Extraction — {extracted.get('patient_name', 'Unknown')} ({REPORT_DATE})"
body = build_email_body(extracted)

print("Preview of email body:")
print("=" * 50)
print(body)
print("=" * 50)

# TODO: uncomment the line below once your send_email function works
# send_email(RECIPIENT_EMAIL, subject, body)

---
## Section 9: Markdown Summary

| Item | Value |
|------|-------|
| **Purpose** | Extract structured clinical data from a discharge note using GPT-4.1-mini, then email the result |
| **Model** | gpt-4.1-mini (OpenAI) |
| **Input** | 1 synthetic KHCC discharge summary |
| **Output** | `extracted_patient_data.csv` + email report |
| **Fields extracted** | 15 (name, MRN, age, gender, diagnosis, stage, 4 labs, meds, allergies, procedures, disposition, follow-up) |
| **Email** | Gmail SMTP with TLS, App Password auth |
| **Author** | [Your Name] |